In [16]:
from pathlib import Path
import re
import json
from datetime import datetime
import pandas as pd

# === a. 파일 경로 구성 ===
BASE_DIR = Path(r"C:\Users\user\Desktop\RAW_LOG")  # 필요시 이 줄만 수정

MIDDLE_FOLDERS = ["TC6", "TC7", "TC8", "TC9", "Vision3"]
TARGET_FOLDERS = ["GoodFile", "BadFile"]  # yyyymmdd\GoodFile / yyyymmdd\BadFile

# === b. 설비 분류 ===
TC_TO_FCT = {
    "TC6": "FCT1",
    "TC7": "FCT2",
    "TC8": "FCT3",
    "TC9": "FCT4",
}

def read_text_file(path: Path) -> list[str]:
    """텍스트 파일을 읽어 줄 단위 리스트로 반환 (인코딩 자동 처리 시도)."""
    for enc in ("cp949", "utf-8-sig", "utf-8"):
        try:
            with path.open("r", encoding=enc, errors="replace") as f:
                return [line.rstrip("\n\r") for line in f]
        except UnicodeDecodeError:
            continue
    with path.open("rb") as f:
        return f.read().decode("latin1", errors="replace").splitlines()

def parse_colon_line(line: str):
    """
    'Key       :Value' 형태를 'Key'(좌측), 'Value'(우측)로 분리해서 반환.
    좌우 공백은 strip.
    """
    if ":" not in line:
        return line.strip(), ""
    left, right = line.split(":", 1)
    return left.strip(), right.strip()

def parse_end_time_fct(line: str):
    """
    FCT용 End Time 파싱
    예) 'End Time :2025/10/01  01:46:41'
    """
    _, value = parse_colon_line(line)
    m = re.search(r"(?P<date>\d{4}/\d{2}/\d{2})\s+(?P<time>\d{2}:\d{2}:\d{2})", value)
    if not m:
        return "", ""
    day_raw = m.group("date")       # '2025/10/01'
    time_raw = m.group("time")      # '01:46:41'
    day = day_raw.replace("/", "")  # '20251001'
    return day, time_raw

def parse_end_time_vision(line: str):
    """
    Vision용 End Time 파싱
    예) 'End Time : 2025-10-01 04:30:55'
    """
    _, value = parse_colon_line(line)
    m = re.search(r"(?P<date>\d{4}-\d{2}-\d{2})\s+(?P<time>\d{2}:\d{2}:\d{2})", value)
    if not m:
        return "", ""
    day_raw = m.group("date")       # '2025-10-01'
    time_raw = m.group("time")      # '04:30:55'
    day = day_raw.replace("-", "")  # '20251001'
    return day, time_raw

step_pattern = re.compile(
    r"""^
    (?P<step_no>\d+\.\d+)\s+
    (?P<desc>.+?)\s*,\s*
    (?P<value>[^,]+)\s*,\s*
    (?P<min>[^,]+)\s*,\s*
    (?P<max>[^,]+)\s*,\s*
    \[(?P<result>[^\]]+)\]
    """,
    re.VERBOSE,
)

def parse_steps(lines: list[str], start_idx: int = 18):
    """
    19번째 줄(인덱스 18)부터 끝까지 step 파싱.
    예:
    '1.01 Test Input Voltage(V) , 14.67, 14.60, 14.80, [PASS]'
    """
    steps = []
    for line in lines[start_idx:]:
        line = line.strip()
        if not line:
            continue
        m = step_pattern.match(line)
        if not m:
            continue
        step_no = m.group("step_no").strip()
        desc = m.group("desc").strip()
        value = m.group("value").strip()
        min_v = m.group("min").strip()
        max_v = m.group("max").strip()
        result = m.group("result").strip()

        step_dict = {
            step_no: desc,          # 1. key : value -> "1.01" : "Test Input Voltage(V)"
            "value": value,         # 2. key : value -> value : "14.67"
            "min": min_v,           # 3. min : "14.60"
            "max": max_v,           # 4. max : "14.80"
            "step result": result,  # 5. step result : "PASS"
        }
        steps.append(step_dict)
    return steps

def classify_equipment(middle_folder: str, lines: list[str]):
    """
    TC6~9 -> FCT1~4 매핑
    Vision3 -> 6번째 줄 Test Program 기준 Vision1/2
    """
    if middle_folder in TC_TO_FCT:
        return TC_TO_FCT[middle_folder]

    if middle_folder == "Vision3":
        # 6번째 줄 (index 5)에 Test Program이 있다고 가정
        if len(lines) >= 6:
            key, value = parse_colon_line(lines[5])
            if "LED1" in value:
                return "Vision1"
            if "LED2" in value:
                return "Vision2"
        return "Vision_Unknown"

    return "Unknown"

def parse_one_log_file(path: Path, middle_folder: str):
    """
    한 개 로그 파일을 파싱해서 JSON용 딕셔너리 + DataFrame 요약용 레코드 반환.
    Vision / FCT 조건을 분리 적용.
    """
    lines = read_text_file(path)

    if len(lines) < 19:
        return None  # 너무 짧으면 스킵

    # 설비 분류 (FCT1~4, Vision1~2 등)
    equip_group = classify_equipment(middle_folder, lines)

    # === Station / End Time 분기 ===
    if equip_group.startswith("FCT"):
        # FCT1~4: 3번째 줄 Station 사용, End Time은 / 형식
        station_key, station_val = parse_colon_line(lines[2])
        end_day, end_time = parse_end_time_fct(lines[8])
    elif equip_group.startswith("Vision"):
        # Vision1~2: 3번째 줄 데이터는 무시, End Time은 - 형식
        station_key, station_val = "", ""   # 3번째 줄 무시
        end_day, end_time = parse_end_time_vision(lines[8])
    else:
        # 예외적인 경우: FCT 스타일로 시도
        station_key, station_val = parse_colon_line(lines[2])
        end_day, end_time = parse_end_time_fct(lines[8])

    # Barcode (5번째 줄)
    barcode_key, barcode_val = parse_colon_line(lines[4])

    # Result (13번째 줄)
    result_key, result_val = parse_colon_line(lines[12])

    # Run Time (14번째 줄)
    runtime_key, runtime_val = parse_colon_line(lines[13])

    # step 파싱 (19번째 줄부터)
    steps = parse_steps(lines, start_idx=18)

    # JSON 구조 (모든 값 문자열)
    json_data = {
        "End day": end_day,                      # yyyyMMdd
        "End time": end_time,                    # hh:mi:ss
        "Station": station_val,                  # FCT의 Station, Vision은 ""
        "Barcode information": barcode_val,      # 전체 바코드 문자열
        "Result": result_val,
        "Run Time": runtime_val,
        "equipment_group": equip_group,          # FCT1~4, Vision1~2
        "equipment_raw": middle_folder,          # TC6/TC7/.../Vision3
        "file_path": str(path),
        "steps": steps,
    }

    # DataFrame 요약용 레코드
    record = {
        "equipment_group": equip_group,
        "equipment_raw": middle_folder,
        "file_path": str(path),
        "End day": end_day,
        "End time": end_time,
        "Station": station_val,
        "Barcode information": barcode_val,
        "Result": result_val,
        "Run Time": runtime_val,
    }

    return json_data, record

# === 메인 스캔 & 파싱 ===
all_json_objects = []
records = []

for mid in MIDDLE_FOLDERS:
    mid_dir = BASE_DIR / mid
    if not mid_dir.exists():
        continue

    # yyyymmdd 폴더 순회
    for date_dir in mid_dir.iterdir():
        if not date_dir.is_dir():
            continue

        for sub in TARGET_FOLDERS:
            target_dir = date_dir / sub
            if not target_dir.exists():
                continue

            for txt_path in target_dir.glob("*.txt"):
                parsed = parse_one_log_file(txt_path, mid)
                if parsed is None:
                    continue
                json_obj, rec = parsed
                all_json_objects.append(json_obj)
                records.append(rec)

# === c 단계 결과를 DataFrame으로 정리 ===
df = pd.DataFrame(records)

if not df.empty:
    df["End_datetime"] = pd.to_datetime(
        df["End day"] + " " + df["End time"],
        format="%Y%m%d %H:%M:%S",
        errors="coerce"
    )

# FCT1~4별, Vision1~2별 최신 10개씩 보기
fct_groups = ["FCT1", "FCT2", "FCT3", "FCT4"]
vision_groups = ["Vision1", "Vision2"]

fct_top10_by_group = {}
vision_top10_by_group = {}

if not df.empty:
    for g in fct_groups:
        sub = df[df["equipment_group"] == g].copy()
        if not sub.empty:
            sub = sub.sort_values("End_datetime", ascending=False).head(10)
            fct_top10_by_group[g] = sub

    for g in vision_groups:
        sub = df[df["equipment_group"] == g].copy()
        if not sub.empty:
            sub = sub.sort_values("End_datetime", ascending=False).head(10)
            vision_top10_by_group[g] = sub

print(f"총 파싱된 파일 수: {len(records)}")
df.head()

총 파싱된 파일 수: 29282


,equipment_group,equipment_raw,file_path,End day,End time,Station,Barcode information,Result,Run Time,End_datetime
0,FCT1,TC6,C:\Users\user\Desktop\RAW_LOG\TC6\20251001\Goo...,20251001,01:46:41,FCT1,BA1WJ25273503681SJ8T-14F014-AE,PASS,27.0,2025-10-01 01:46:41
1,FCT1,TC6,C:\Users\user\Desktop\RAW_LOG\TC6\20251001\Goo...,20251001,01:46:05,FCT1,BA1WJ25273503683SJ8T-14F014-AE,PASS,27.8,2025-10-01 01:46:05
2,FCT1,TC6,C:\Users\user\Desktop\RAW_LOG\TC6\20251001\Goo...,20251001,01:45:29,FCT1,BA1WJ25273503685SJ8T-14F014-AE,PASS,26.3,2025-10-01 01:45:29
3,FCT1,TC6,C:\Users\user\Desktop\RAW_LOG\TC6\20251001\Goo...,20251001,01:44:54,FCT1,BA1WJ25273503687SJ8T-14F014-AE,PASS,26.3,2025-10-01 01:44:54
4,FCT1,TC6,C:\Users\user\Desktop\RAW_LOG\TC6\20251001\Goo...,20251001,01:48:39,FCT1,BA1WJ25273503689SJ8T-14F014-AE,PASS,28.1,2025-10-01 01:48:39


In [17]:
# FCT1~4별 상위 10개 확인
for g, subdf in fct_top10_by_group.items():
    print("====", g, "====")
    display(subdf[["End day", "End time", "Station", "Barcode information", "Result", "Run Time", "file_path"]])

# Vision1~2별 상위 10개 확인
for g, subdf in vision_top10_by_group.items():
    print("====", g, "====")
    display(subdf[["End day", "End time", "Station", "Barcode information", "Result", "Run Time", "file_path"]])

==== FCT1 ====


,End day,End time,Station,Barcode information,Result,Run Time,file_path
3403,20251003,07:44:58,FCT1,BA1WJ25276500337SJ8T-14F014-AE,FAIL,20.1,C:\Users\user\Desktop\RAW_LOG\TC6\20251003\Goo...
3404,20251003,07:44:29,FCT1,BA1WJ25276500339SJ8T-14F014-AE,PASS,26.5,C:\Users\user\Desktop\RAW_LOG\TC6\20251003\Goo...
3405,20251003,07:43:53,FCT1,BA1WJ25276500341SJ8T-14F014-AE,PASS,26.2,C:\Users\user\Desktop\RAW_LOG\TC6\20251003\Goo...
3399,20251003,07:43:18,FCT1,BA1WJ25276500329SJ8T-14F014-AE,PASS,26.6,C:\Users\user\Desktop\RAW_LOG\TC6\20251003\Goo...
3400,20251003,07:42:43,FCT1,BA1WJ25276500331SJ8T-14F014-AE,PASS,28.5,C:\Users\user\Desktop\RAW_LOG\TC6\20251003\Goo...
3401,20251003,07:42:06,FCT1,BA1WJ25276500333SJ8T-14F014-AE,PASS,28.4,C:\Users\user\Desktop\RAW_LOG\TC6\20251003\Goo...
3402,20251003,07:41:29,FCT1,BA1WJ25276500335SJ8T-14F014-AE,PASS,28.9,C:\Users\user\Desktop\RAW_LOG\TC6\20251003\Goo...
3396,20251003,07:40:15,FCT1,BA1WJ25276500324SJ8T-14F014-AE,PASS,27.3,C:\Users\user\Desktop\RAW_LOG\TC6\20251003\Goo...
3397,20251003,07:39:39,FCT1,BA1WJ25276500326SJ8T-14F014-AE,PASS,32.4,C:\Users\user\Desktop\RAW_LOG\TC6\20251003\Goo...
3398,20251003,07:38:58,FCT1,BA1WJ25276500328SJ8T-14F014-AE,PASS,32.3,C:\Users\user\Desktop\RAW_LOG\TC6\20251003\Goo...


==== FCT2 ====


,End day,End time,Station,Barcode information,Result,Run Time,file_path
6957,20251003,07:45:25,FCT2,BA1WJ25276500336SJ8T-14F014-AE,PASS,25.9,C:\Users\user\Desktop\RAW_LOG\TC7\20251003\Goo...
6958,20251003,07:44:50,FCT2,BA1WJ25276500338SJ8T-14F014-AE,PASS,27.3,C:\Users\user\Desktop\RAW_LOG\TC7\20251003\Goo...
6959,20251003,07:44:13,FCT2,BA1WJ25276500340SJ8T-14F014-AE,PASS,26.1,C:\Users\user\Desktop\RAW_LOG\TC7\20251003\Goo...
6960,20251003,07:43:38,FCT2,BA1WJ25276500342SJ8T-14F014-AE,PASS,27.2,C:\Users\user\Desktop\RAW_LOG\TC7\20251003\Goo...
6954,20251003,07:43:02,FCT2,BA1WJ25276500330SJ8T-14F014-AE,PASS,29.0,C:\Users\user\Desktop\RAW_LOG\TC7\20251003\Goo...
6955,20251003,07:42:24,FCT2,BA1WJ25276500332SJ8T-14F014-AE,PASS,27.3,C:\Users\user\Desktop\RAW_LOG\TC7\20251003\Goo...
6956,20251003,07:41:47,FCT2,BA1WJ25276500334SJ8T-14F014-AE,PASS,26.2,C:\Users\user\Desktop\RAW_LOG\TC7\20251003\Goo...
6950,20251003,07:41:12,FCT2,BA1WJ25276500322SJ8T-14F014-AE,PASS,26.5,C:\Users\user\Desktop\RAW_LOG\TC7\20251003\Goo...
6951,20251003,07:40:29,FCT2,BA1WJ25276500323SJ8T-14F014-AE,PASS,26.6,C:\Users\user\Desktop\RAW_LOG\TC7\20251003\Goo...
6952,20251003,07:39:49,FCT2,BA1WJ25276500325SJ8T-14F014-AE,PASS,28.0,C:\Users\user\Desktop\RAW_LOG\TC7\20251003\Goo...


==== FCT3 ====


...

,End day,End time,Station,Barcode information,Result,Run Time,file_path
14814,20251003,07:57:30,FCT4,BA1WJ25276500236SJ8T-14F014-AE,PASS,25.9,C:\Users\user\Desktop\RAW_LOG\TC9\20251003\Goo...
14960,20251003,07:56:56,FCT4,BA1WJ25276502161SJ8T-14F014-AE,PASS,25.6,C:\Users\user\Desktop\RAW_LOG\TC9\20251003\Goo...
14984,20251003,07:56:23,FCT4,BA1WJ25276502318SJ8T-14F014-AE,PASS,31.9,C:\Users\user\Desktop\RAW_LOG\TC9\20251003\Goo...
14884,20251003,07:55:40,FCT4,BA1WJ25276501484SJ8T-14F014-AE,PASS,25.5,C:\Users\user\Desktop\RAW_LOG\TC9\20251003\Goo...
14983,20251003,07:55:07,FCT4,BA1WJ25276502316SJ8T-14F014-AE,PASS,33.3,C:\Users\user\Desktop\RAW_LOG\TC9\20251003\Goo...
14982,20251003,07:54:25,FCT4,BA1WJ25276502314SJ8T-14F014-AE,PASS,27.5,C:\Users\user\Desktop\RAW_LOG\TC9\20251003\Goo...
14981,20251003,07:53:50,FCT4,BA1WJ25276502312SJ8T-14F014-AE,PASS,27.5,C:\Users\user\Desktop\RAW_LOG\TC9\20251003\Goo...
14791,20251003,07:53:14,FCT4,BA1WJ25276500144SJ8T-14F014-AE,PASS,33.2,C:\Users\user\Desktop\RAW_LOG\TC9\20251003\Goo...
14789,20251003,07:52:33,FCT4,BA1WJ25276500109SJ8T-14F014-AE,PASS,31.7,C:\Users\user\Desktop\RAW_LOG\TC9\20251003\Goo...
14872,20251003,07:51:53,FCT4,BA1WJ25276501318SJ8T-14F014-AE,PASS,25.6,C:\Users\user\Desktop\RAW_LOG\TC9\20251003\Goo...


==== Vision1 ====


,End day,End time,Station,Barcode information,Result,Run Time,file_path
28715,20251003,07:55:27,,BA1WJ25276500336SJ8T-14F014-AE,PASS,1.843073s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
28728,20251003,07:55:03,,BA1WJ25276501220SJ8T-14F014-AE,PASS,1.8350941s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
28741,20251003,07:54:50,,BA1WJ25276501233SJ8T-14F014-AE,PASS,1.8490567s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
28729,20251003,07:54:35,,BA1WJ25276501221SJ8T-14F014-AE,PASS,1.8410784s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
28730,20251003,07:54:22,,BA1WJ25276501222SJ8T-14F014-AE,PASS,1.8430708s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
28731,20251003,07:54:07,,BA1WJ25276501223SJ8T-14F014-AE,PASS,1.8690043s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
28727,20251003,07:53:54,,BA1WJ25276501219SJ8T-14F014-AE,PASS,1.8510525s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
28564,20251003,07:53:40,,BA1WJ25276500159SJ8T-14F014-AE,PASS,1.8350951s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
28740,20251003,07:53:26,,BA1WJ25276501232SJ8T-14F014-AE,PASS,1.8610252s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
28569,20251003,07:53:12,,BA1WJ25276500164SJ8T-14F014-AE,PASS,1.8749879s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...


==== Vision2 ====


,End day,End time,Station,Barcode information,Result,Run Time,file_path
28639,20251003,07:57:48,,BA1WJ25276500236SJ8T-14F014-AE,PASS,1.8319712s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
29193,20251003,07:57:34,,BA1WJ25276502012SJ8T-14F014-AE,PASS,1.8338404s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
29213,20251003,07:57:14,,BA1WJ25276502161SJ8T-14F014-AE,PASS,1.8353588s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
29280,20251003,07:56:59,,BA1WJ25276502317SJ8T-14F014-AE,PASS,1.8191403s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
29281,20251003,07:56:40,,BA1WJ25276502318SJ8T-14F014-AE,PASS,1.8530482s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
29273,20251003,07:56:23,,BA1WJ25276502310SJ8T-14F014-AE,PASS,1.8331012s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
28874,20251003,07:56:00,,BA1WJ25276501484SJ8T-14F014-AE,PASS,1.851051s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
29278,20251003,07:55:46,,BA1WJ25276502315SJ8T-14F014-AE,PASS,1.8650186s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
29279,20251003,07:55:24,,BA1WJ25276502316SJ8T-14F014-AE,PASS,1.843071s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
29276,20251003,07:55:10,,BA1WJ25276502313SJ8T-14F014-AE,PASS,1.8370873s,C:\Users\user\Desktop\RAW_LOG\Vision3\20251003...
